# First LLM call with `aai-core`

This notebook demonstrates the smallest useful path from a cloned SDK repository to a configured model call. It uses keyless Azure CLI authentication and a logical model name; it never stores a PAT, client secret, or API key.

Convention: Jupyter notebooks (like this one) are for **local exploration**; anything that deploys through CD uses Databricks-format `.py` notebooks inside a generated project. Neither kind hardcodes configuration — `bootstrap()` discovers `aai-platform.yml` (upward search, or the `AAI_PLATFORM_CONFIG` env var).

Before starting Jupyter from the repository root, run:

```bash
python -m venv .venv
source .venv/bin/activate
python -m pip install -e '.[databricks]'
az login
export DATABRICKS_HOST=<workspace host — see platform-identifiers.json>
export DATABRICKS_AUTH_TYPE=azure-cli
cp aai-platform.example.yml aai-platform.yml
jupyter lab
```

Replace `replace-with-serving-endpoint` in `aai-platform.yml` with a Databricks model serving endpoint on which your identity has `CAN_QUERY`.

In [ ]:
import os
from pathlib import Path

config_path = Path("aai-platform.yml")
if not config_path.is_file():
    raise FileNotFoundError(
        "Copy aai-platform.example.yml to aai-platform.yml and configure the "
        "general-chat deployment before running this notebook."
    )

if not os.getenv("DATABRICKS_HOST"):
    raise RuntimeError("Set DATABRICKS_HOST before starting Jupyter.")

print(
    {
        "config": str(config_path),
        "databricks_host_configured": True,
        "databricks_auth_type": os.getenv("DATABRICKS_AUTH_TYPE", "default"),
    }
)

## Resolve the logical model

`bootstrap()` loads the platform configuration. Application code asks for `general-chat`; the environment configuration decides which concrete endpoint serves it.

In [ ]:
from aai_core import bootstrap

ctx = bootstrap(config_path)
model = ctx.providers.model("general-chat")

print(
    {
        "application": ctx.tags.application,
        "environment": ctx.tags.environment,
        "logical_model": model.logical_name,
        "provider": model.provider,
        "deployment": model.model,
    }
)

## Make the LLM call

The stable adapter returns normalized content, model identity, latency, token usage, and tool calls. Provider-specific functionality remains available through `model.native_client` when needed.

In [ ]:
try:
    response = model.generate(
        [
            {
                "role": "user",
                "content": "Explain Unity Catalog in three concise sentences.",
            }
        ],
        temperature=0.2,
        max_tokens=200,
    )
except Exception as exc:
    message = str(exc)
    if "403" in message or "PERMISSION_DENIED" in message:
        raise RuntimeError(
            "Permission denied: your identity lacks CAN_QUERY on the serving "
            "endpoint that aai-platform.yml maps to general-chat. Ask the "
            "platform team for CAN_QUERY on that endpoint, then rerun this cell."
        ) from exc
    if (
        "404" in message
        or "RESOURCE_DOES_NOT_EXIST" in message
        or "NotFound" in message
    ):
        raise RuntimeError(
            "Endpoint not found: the deployment configured for general-chat in "
            "aai-platform.yml does not exist in this workspace. List available "
            "endpoints with `databricks serving-endpoints list` and update the "
            "deployment value."
        ) from exc
    raise

print(response.content)
print(
    {
        "provider": response.provider,
        "model": response.model,
        "latency_ms": round(response.latency_ms, 1),
        "usage": dict(response.usage),
    }
)

## Source checkout versus released wheel

This notebook uses an editable install because it lives in the SDK source repository. A generated application repository instead pins an immutable `aai-core` version.

- The `publish-sdk` workflow builds `aai_core-<version>-py3-none-any.whl`, writes its SHA-256 checksum, and uploads both to `/Volumes/dbx_dev/dbx_platform/python_packages/aai_core/<version>/`.
- A generated project's `scripts/install_core.py` downloads that exact wheel and checksum for local development, verifies the digest, and installs it.
- Its Databricks job installs the same wheel directly from the Unity Catalog volume.
- The generated application itself is a separate wheel built from its own `src/app` package.

The volume is therefore the internal immutable SDK package store; it is not required while actively developing this SDK from a clone.